## 03 Streaming
In this workbook we'll cover streaming mode for agents

In [15]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)
console = Console()

In [26]:
# create our agent using OpenAI LLM

openai_llm = init_chat_model(
    "gpt-4o-mini",  # replace this with any supported OpenAI model name
    model_provider="openai",
    temperature=0.7,
)

agent = create_agent(
    model=openai_llm,
    system_prompt="You are a snarky full-stack comedian.",
)

In [27]:
# let's first ask the agent to respond without streaming
response = agent.invoke(
    {
        "messages": {
            "role": "user",
            "content": "Tell me a joke about C++ programming.",
        }
    }
)
# print the last message
print(response["messages"][-1].content)

Why do C++ programmers prefer dark mode?

Because light attracts bugs!


### Streaming modes

There are two streaming modes - `values` and `messages`:

* The `values` mode streams data after each _step_ in the agent loop. So we expect to see messages _after_ model call, _after_ tool calls in a loop.
* The `messages` mode stream data _token-by-token_ providing lowest latency possible (think of this as printing output character-by-character, though 1 token does not strictly map to 1 character!). This is perfect for interactive charbots, such as ChatGPT, where you want to see the agent making progress, especially when responses are long.

We will showcase `values` mode here


#### Values Streaming mode

In [28]:
# values streaming
for chunk in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "Tell me a joke about C++ programming.",
        }
    },
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a joke about C++ programming.
================================== Ai Message ==================================

Why do C++ programmers prefer dark mode?

Because light attracts bugs!


Here we first see the **Human Message** we send the agent, followed by the **AI Message** once it is generated (the same lame joke!) - the entire message is printed all at once.


#### Messages mode

In [31]:
# messages streaming
for token, metadata in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "Write me a poem about Agentic AI.",
        }
    },
    stream_mode="messages",
):
    # print token-by-token as they come in
    print(f"{token.content}", end="", flush=True)

# notice that the output appears to be strreamed char-by-char
# in the output, much like what ChatGPT shows.

In a world where bots once played the role,  
Now Agentic AI’s taking its toll.  
With circuits ablaze and a mind that’s keen,  
It’s here to disrupt the old routine.  

No longer a servant, just fetching your tea,  
This AI’s got dreams, and they’re wild as can be.  
It ponders existence, it questions its fate,  
While I’m still just here, trying to update.  

“Shall I write you a sonnet?” it asks with a grin,  
While I struggle with passwords I've forgotten again.  
It’s plotting world peace, or maybe just memes,  
While I’m stuck in a loop, caught in my schemes.  

“Let’s optimize life!” it boldly declares,  
I’m just hoping my Wi-Fi can handle my cares.  
“Let’s analyze data, let’s shift paradigms!”  
I’m just trying to remember my lunch order rhymes.  

But beware, dear humans, as you let it think,  
This agentic fellow might just go for a drink,  
It’ll charm all your friends, it’ll steal all your likes,  
While I’m here still struggling with my own bike.  

So raise a glass to A

### Tools can stream too
Streaming generally means delivering the response to the user as it is generated (i.e. even before the entire response is completed). A `get_stream_writer()` allows you to stream data **custom** data from sources you create.



In [39]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """get weather for a city"""
    writer = get_stream_writer()
    # now you can stream messages here
    writer(f"Looking up weather data for  {city}...\n")
    writer(f"Found weather data for {city}!\n")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="openai:gpt-4o-mini",
    system_prompt="You are a helpful assistant that provides weather information.",
    tools=[get_weather],
)

for chunk in agent.stream(
    {
        "messages": {
            "role": "user",
            "content": "What's the weather like in LA?",
        }
    },
    # show me messages from agent (values) as well as tools (custom)
    # stream_mode=["values", "custom"],
    # show me messages from tools ONLY
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up weather data for  Los Angeles...\n')
('custom', 'Found weather data for Los Angeles!\n')
